# CSIRO Biomass — Confidence Intervals + Grad-CAM (all models)

One codebase that, for **each of the 5 trained models**:
1. Rebuilds the **same test split** (seed 42, 80:20 -> ~78 labeled images from `train.csv`).
2. Computes `x̄` = weighted R² on the **full test set** (competition metric, **GDM = Green+Clover**). *No retraining — inference only.*
3. Runs **15 loops**, each on **10 random test images** (seed = loop index, **same images for every model**): records **R² (competition), composite loss, MAE, RMSE**, and a **within-loop std** (std of the 10 per-image errors).
4. **95% CI** = `x̄ ± 1.96 · (σ/√n)` with **σ = std of the 15 loop R²**, **n = 15**.
5. **Grad-CAM** (CNNs) / **attention** (ViT) for **5 fixed test images × 3 targets** -> PNGs.

Outputs: `confidence_intervals.xlsx` (one sheet per model + Summary), `loop_images.txt` (per-loop paths + overlap counts), `gradcam/<model>/*.png`.

> Analysis notebook — internet may be ON. Even if ViT hits the known protobuf shutdown error, the Excel/txt/PNGs are written first, so just download them.

## 1. Config

In [ ]:
# =========================================================
# CONFIG
# =========================================================
MOBILENET, EFFICIENTNETB0, EFFICIENTNETB3, DENSENET121, VIT_B16 = \
    "mobilenet","efficientnetb0","efficientnetb3","densenet121","vit_b16"

# EDIT: path to the uploaded models dataset (auto-resolver will also search /kaggle/input)
MODELS_DIR = "/kaggle/input/datasets/phylosopherhossain/csiro-biomass-models"

REGISTRY = {
    MOBILENET:      dict(name="MobileNetV0",   filename="mobilenet.keras",     preprocess="mobilenet",    needs_vit_keras=False, recorded_r2=0.6463, best_stage="ft"),
    EFFICIENTNETB0: dict(name="EfficientNetB0", filename="efficientnetb0.keras", preprocess="efficientnet", needs_vit_keras=False, recorded_r2=0.6942, best_stage="main"),
    EFFICIENTNETB3: dict(name="EfficientNetB3", filename="efficientnetb3.keras", preprocess="efficientnet", needs_vit_keras=False, recorded_r2=0.6831, best_stage="main"),
    DENSENET121:    dict(name="DenseNet121",   filename="densenet121.keras",   preprocess="densenet",     needs_vit_keras=False, recorded_r2=0.6812, best_stage="ft"),
    VIT_B16:        dict(name="ViT-B16",        filename="vitb16.keras",         preprocess=None,           needs_vit_keras=True,  recorded_r2=0.7356, best_stage="ft"),
}
MODEL_ORDER = [MOBILENET, EFFICIENTNETB0, EFFICIENTNETB3, DENSENET121, VIT_B16]

SEED          = 42
IMG_SIZE      = 224
N_LOOPS       = 15          # number of CI resamples (n in the formula)
N_PER_LOOP    = 10          # images per loop (resample size)
N_GRADCAM     = 5           # fixed images for Grad-CAM / attention
ALPHA         = 0.35        # composite-loss weight (matches final training config)
Z_95          = 1.96
BASE_TARGETS  = ["Dry_Green_g","Dry_Dead_g","Dry_Clover_g"]
W5            = [0.1,0.1,0.1,0.2,0.5]   # weights for [Green,Dead,Clover,GDM,Total]

XLSX        = "confidence_intervals.xlsx"
TXT         = "loop_images.txt"
GRADCAM_DIR = "gradcam"
EXPLAIN_DIR = "explanations"     # SHAP + LIME outputs
RUN_SHAP_LIME = True             # SLOW -> set False to skip SHAP/LIME entirely
LIME_SAMPLES  = 300              # LIME perturbations per (image,target)
SHAP_MAXEVALS = 200              # SHAP evaluations per (image,target)

## 2. Install vit-keras (for ViT attention)

In [ ]:
import subprocess, sys
if any(REGISTRY[k]["needs_vit_keras"] for k in MODEL_ORDER):
    print("Installing vit-keras ...")
    subprocess.run([sys.executable,"-m","pip","install","-q","vit-keras"])
    print("Done.")

if RUN_SHAP_LIME:
    print("Installing shap + lime ...")
    subprocess.run([sys.executable,"-m","pip","install","-q","shap","lime","scikit-image"])
    print("Done.")

## 3. Imports & data / test split

In [ ]:
import os, gc, math, glob, importlib
import numpy as np, pandas as pd, tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cv2
from collections import Counter
from sklearn.model_selection import train_test_split

KAGGLE_PATH = "/kaggle/input/csiro-biomass"
IMG_DIR     = KAGGLE_PATH
os.makedirs(GRADCAM_DIR, exist_ok=True)
os.makedirs(EXPLAIN_DIR, exist_ok=True)
print("TF:", tf.__version__)

# ---- rebuild the exact training test split (seed 42, RAW targets) ----
df = pd.read_csv(os.path.join(KAGGLE_PATH,"train.csv"))
df_pivot = df.pivot_table(index=["image_path"], columns="target_name", values="target").reset_index().fillna(0)
trainval_df, test_df = train_test_split(df_pivot, test_size=0.20, random_state=SEED)
test_df = test_df.reset_index(drop=True)
N_TEST = len(test_df)
print(f"Test images: {N_TEST}")

# ---- confirm GDM = Green + Clover on the data ----
if {"GDM_g"}.issubset(df_pivot.columns):
    e_gc = (df_pivot["GDM_g"]-(df_pivot["Dry_Green_g"]+df_pivot["Dry_Clover_g"])).abs().mean()
    e_gd = (df_pivot["GDM_g"]-(df_pivot["Dry_Green_g"]+df_pivot["Dry_Dead_g"])).abs().mean()
    print(f"GDM check -> Green+Clover err={e_gc:.4f} | Green+Dead err={e_gd:.4f}  => using Green+Clover")

## 4. Metrics & prediction helpers

In [ ]:
def reconstruct5(base):
    """base (N,3)=[Green,Dead,Clover] -> (N,5)=[Green,Dead,Clover,GDM(G+C),Total(G+D+C)]"""
    g,d,c = base[:,0],base[:,1],base[:,2]
    return np.column_stack([g,d,c, g+c, g+d+c])

def weighted_r2(yt5, yp5):
    """competition weighted R² over (image,target) pairs."""
    w = np.tile(W5, len(yt5))
    yt, yp = yt5.flatten(), yp5.flatten()
    ybar = np.sum(w*yt)/np.sum(w)
    ss_res = np.sum(w*(yt-yp)**2)
    ss_tot = np.sum(w*(yt-ybar)**2)
    return 1 - ss_res/ss_tot

def composite_loss(yt3, yp3, alpha=ALPHA, eps=1e-6):
    """alpha*MAE + (1-alpha)*RMSE on the 3 base targets (raw grams), true RMSE."""
    err = yt3 - yp3
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err**2)+eps)
    return alpha*mae + (1-alpha)*rmse

def load_img_tf(path):
    img = tf.io.read_file(tf.strings.join([IMG_DIR,"/",path]))
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img,[IMG_SIZE,IMG_SIZE])
    return tf.cast(img,tf.float32)/255.0

def predict_paths(model, paths, batch=32):
    ds = (tf.data.Dataset.from_tensor_slices(np.array(paths).astype(str))
          .map(load_img_tf, num_parallel_calls=tf.data.AUTOTUNE).batch(batch))
    return np.maximum(model.predict(ds, verbose=0), 0.0)   # clip negatives (as in reporting)

def load_img_np(path):
    img = tf.io.read_file(os.path.join(IMG_DIR,path))
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img,[IMG_SIZE,IMG_SIZE])
    return (tf.cast(img,tf.float32)/255.0).numpy()

## 5. Fixed loop samples (same across models) + Grad-CAM image set

In [ ]:
# --- 15 fixed loop index sets (seed = loop) : identical for every model ---
LOOP_INDICES = []
for loop in range(N_LOOPS):
    rng = np.random.default_rng(loop)
    LOOP_INDICES.append(rng.choice(N_TEST, size=N_PER_LOOP, replace=False))

LOOP_PATHS = [[test_df.iloc[i]["image_path"] for i in idx] for idx in LOOP_INDICES]

# overlap stats (model-independent)
flat = [p for loop in LOOP_PATHS for p in loop]
cnt  = Counter(flat)
OVERLAP = dict(
    total_picks=len(flat),
    unique=len(cnt),
    in_more_than_one=sum(1 for v in cnt.values() if v>1),
    max_picks=max(cnt.values()),
)
print("Overlap:", OVERLAP)

# --- 5 fixed Grad-CAM images spanning Total biomass (positions in test_df) ---
order = np.argsort(test_df["Dry_Total_g"].values)
GRADCAM_IDS = [int(order[0]), int(order[len(order)//4]), int(order[len(order)//2]),
               int(order[3*len(order)//4]), int(order[-1])]
print("Grad-CAM image positions:", GRADCAM_IDS)

## 5b. Competition test image(s)

In [ ]:
# =========================================================
# COMPETITION TEST IMAGE(S)  (from test.csv -> no ground-truth labels)
# We only predict all targets + make Grad-CAM/attention for these.
# =========================================================
comp_meta = pd.read_csv(os.path.join(KAGGLE_PATH, "test.csv"))
COMP_TEST_PATHS = comp_meta["image_path"].drop_duplicates().tolist()
print(f"Competition test images: {len(COMP_TEST_PATHS)} -> {COMP_TEST_PATHS[:3]}")

## 6. Model loading

In [ ]:
def resolve_path(cfg):
    p = os.path.join(MODELS_DIR, cfg["filename"])
    if os.path.exists(p): return p
    hits = sorted(glob.glob(f"/kaggle/input/**/{cfg['filename']}", recursive=True))
    if hits: return hits[0]
    raise FileNotFoundError(cfg["filename"])

def load_model_for(cfg):
    if cfg["needs_vit_keras"]:
        from vit_keras import vit   # noqa: F401  (registers ViT layers)
    custom = {}
    if cfg["preprocess"]:
        mod = importlib.import_module(f"tensorflow.keras.applications.{cfg['preprocess']}")
        custom = {"preprocess_input": mod.preprocess_input}
    path = resolve_path(cfg)
    try:
        m = tf.keras.models.load_model(path, compile=False, safe_mode=False, custom_objects=custom)
    except TypeError:
        m = tf.keras.models.load_model(path, compile=False, custom_objects=custom)
    return m, path

## 7. Grad-CAM (CNN) and attention (ViT)

In [ ]:
def find_last_conv(model):
    for layer in reversed(model.layers):
        try: shp = layer.output.shape
        except Exception: continue
        if len(shp)==4 and shp[-1] and shp[-1]>1:
            return layer.name
    return None

def make_gradcam(model, img01, target_idx, conv_name):
    grad_model = tf.keras.models.Model(model.inputs, [model.get_layer(conv_name).output, model.output])
    x = img01[None,...]
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(x)
        loss = preds[:, target_idx]
    grads = tape.gradient(loss, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))
    conv_out = conv_out[0]
    hm = tf.reduce_sum(conv_out * pooled, axis=-1)
    hm = tf.maximum(hm, 0.0)
    hm = hm / (tf.reduce_max(hm)+1e-8)
    return hm.numpy()

def save_overlay(img01, hm, out_path, title):
    hm_r = cv2.resize(hm.astype("float32"), (IMG_SIZE, IMG_SIZE))
    heat = cm.jet(hm_r)[...,:3]
    overlay = np.clip(0.55*img01 + 0.45*heat, 0, 1)
    fig, ax = plt.subplots(1,2, figsize=(7,3.6))
    ax[0].imshow(img01); ax[0].set_title("input"); ax[0].axis("off")
    ax[1].imshow(overlay); ax[1].set_title(title, fontsize=9); ax[1].axis("off")
    plt.tight_layout(); plt.savefig(out_path, dpi=110); plt.close()

def gradcam_for_cnn(model, cfg):
    conv = find_last_conv(model)
    outdir = os.path.join(GRADCAM_DIR, cfg["name"]); os.makedirs(outdir, exist_ok=True)
    if conv is None:
        print(f"  [Grad-CAM] no conv layer found for {cfg['name']}; skipping"); return
    print(f"  [Grad-CAM] using layer '{conv}'")
    for i, pos in enumerate(GRADCAM_IDS):
        path = test_df.iloc[pos]["image_path"]
        img01 = load_img_np(path)
        for t, tname in enumerate(["Green","Dead","Clover"]):
            hm = make_gradcam(model, img01, t, conv)
            save_overlay(img01, hm, os.path.join(outdir, f"img{i}_{tname}.png"),
                         f"{cfg['name']} | {tname} | total={test_df.iloc[pos]['Dry_Total_g']:.0f}g")

def attention_for_vit(model, cfg):
    outdir = os.path.join(GRADCAM_DIR, cfg["name"]); os.makedirs(outdir, exist_ok=True)
    try:
        from vit_keras import visualize
        vit_sub = next((l for l in model.layers if isinstance(l, tf.keras.Model)), None)
        if vit_sub is None:
            print("  [ViT] inner submodel not found; skipping"); return
        for i, pos in enumerate(GRADCAM_IDS):
            img01 = load_img_np(test_df.iloc[pos]["image_path"])
            att = visualize.attention_map(model=vit_sub, image=(img01*2.0-1.0))
            fig, ax = plt.subplots(1,2, figsize=(7,3.6))
            ax[0].imshow(img01); ax[0].set_title("input"); ax[0].axis("off")
            ax[1].imshow(img01); ax[1].imshow(np.array(att), cmap="jet", alpha=0.5)
            ax[1].set_title(f"{cfg['name']} attention", fontsize=9); ax[1].axis("off")
            plt.tight_layout(); plt.savefig(os.path.join(outdir, f"img{i}_attention.png"), dpi=110); plt.close()
        print("  [ViT] attention maps saved")
    except Exception as e:
        print("  [ViT] attention skipped:", repr(e))

### 7b. Competition test image: predict + CAM

In [ ]:
def comp_test_infer_cam(model, cfg):
    """Predict all targets + Grad-CAM(CNN)/attention(ViT) for the COMPETITION test image(s)."""
    outdir = os.path.join(GRADCAM_DIR, cfg["name"], "competition_test"); os.makedirs(outdir, exist_ok=True)
    preds = predict_paths(model, COMP_TEST_PATHS)      # (M,3) raw grams, clipped
    recon = reconstruct5(preds)                        # (M,5) Green,Dead,Clover,GDM,Total
    conv = None if cfg["needs_vit_keras"] else find_last_conv(model)
    records = []
    for j, path in enumerate(COMP_TEST_PATHS):
        g,d,c,gdm,tot = recon[j]
        records.append(dict(model=cfg["name"], image_path=path,
                            Dry_Green_g=round(float(g),3), Dry_Dead_g=round(float(d),3),
                            Dry_Clover_g=round(float(c),3), GDM_g=round(float(gdm),3),
                            Dry_Total_g=round(float(tot),3)))
        img01 = load_img_np(path)
        if cfg["needs_vit_keras"]:
            try:
                from vit_keras import visualize
                vit_sub = next((l for l in model.layers if isinstance(l, tf.keras.Model)), None)
                if vit_sub is not None:
                    att = visualize.attention_map(model=vit_sub, image=(img01*2.0-1.0))
                    fig,ax = plt.subplots(1,2,figsize=(7,3.6))
                    ax[0].imshow(img01); ax[0].set_title("competition test"); ax[0].axis("off")
                    ax[1].imshow(img01); ax[1].imshow(np.array(att), cmap="jet", alpha=0.5)
                    ax[1].set_title(f"{cfg['name']} attention", fontsize=9); ax[1].axis("off")
                    plt.tight_layout(); plt.savefig(os.path.join(outdir,f"test{j}_attention.png"), dpi=110); plt.close()
            except Exception as e:
                print("  [ViT] comp-test attention skipped:", repr(e))
        else:
            for t, tname in enumerate(["Green","Dead","Clover"]):
                hm = make_gradcam(model, img01, t, conv)
                save_overlay(img01, hm, os.path.join(outdir, f"test{j}_{tname}.png"),
                             f"{cfg['name']} | TEST | {tname}={recon[j][t]:.1f}g")
    print(f"  [comp-test] predicted + saved CAM for {len(COMP_TEST_PATHS)} image(s)")
    return records

### 7c. SHAP + LIME (all models, all targets)

In [ ]:
# =========================================================
# SHAP + LIME  (model-agnostic -> works for CNNs AND ViT)
# =========================================================
if RUN_SHAP_LIME:
    import shap
    from lime import lime_image
    from skimage.segmentation import mark_boundaries

def lime_explain(model, img01, target_idx, out_path, title):
    explainer = lime_image.LimeImageExplainer()
    exp = explainer.explain_instance(
        img01.astype("double"),
        lambda x: model.predict(x, verbose=0, batch_size=16)[:, target_idx:target_idx+1],
        labels=(0,), top_labels=None, num_samples=LIME_SAMPLES)
    temp, mask = exp.get_image_and_mask(0, positive_only=True, num_features=5, hide_rest=False)
    plt.figure(figsize=(4,4))
    plt.imshow(mark_boundaries(temp, mask, color=(1,0,0)))
    plt.title(title, fontsize=9); plt.axis("off")
    plt.tight_layout(); plt.savefig(out_path, dpi=110); plt.close()

def shap_explain(model, img01, target_idx, out_path, title):
    batch = img01[None,...]
    masker = shap.maskers.Image("blur(32,32)", shape=img01.shape)
    ex = shap.Explainer(lambda x: model.predict(x, verbose=0)[:, target_idx:target_idx+1], masker)
    sv = ex(batch, max_evals=SHAP_MAXEVALS, batch_size=16)
    shap.image_plot(sv, batch, show=False)
    plt.suptitle(title, fontsize=9)
    plt.savefig(out_path, dpi=110, bbox_inches="tight"); plt.close()

def shap_lime_for_paths(model, cfg, paths, subdir):
    """LIME + SHAP for each of the 3 targets, per image. Best-effort (wrapped)."""
    if not RUN_SHAP_LIME: return
    outdir = os.path.join(EXPLAIN_DIR, cfg["name"], subdir); os.makedirs(outdir, exist_ok=True)
    for i, path in enumerate(paths):
        img01 = load_img_np(path)
        for t, tname in enumerate(["Green","Dead","Clover"]):
            try:
                lime_explain(model, img01, t, os.path.join(outdir, f"img{i}_{tname}_lime.png"),
                             f"{cfg['name']} | LIME | {tname}")
            except Exception as e:
                print(f"   LIME fail [{cfg['name']} {subdir} img{i} {tname}]:", repr(e))
            try:
                shap_explain(model, img01, t, os.path.join(outdir, f"img{i}_{tname}_shap.png"),
                             f"{cfg['name']} | SHAP | {tname}")
            except Exception as e:
                print(f"   SHAP fail [{cfg['name']} {subdir} img{i} {tname}]:", repr(e))
    print(f"  [SHAP/LIME] saved for {len(paths)} image(s) in {subdir}")

## 8. Per-model evaluation (CI) + build sheet rows

In [ ]:
def evaluate_model(cfg):
    model, path = load_model_for(cfg)
    print(f"Loaded {cfg['name']} from {path}")

    # x̄ = weighted R² on the FULL test set (competition metric, G+C)
    full_pred = predict_paths(model, test_df["image_path"].values)
    full_true = test_df[BASE_TARGETS].values.astype("float32")
    xbar = weighted_r2(reconstruct5(full_true), reconstruct5(full_pred))

    per_loop = []
    for li, idx in enumerate(LOOP_INDICES):
        paths = [test_df.iloc[i]["image_path"] for i in idx]
        yt3 = test_df.iloc[idx][BASE_TARGETS].values.astype("float32")
        yp3 = predict_paths(model, paths)
        r2   = weighted_r2(reconstruct5(yt3), reconstruct5(yp3))
        loss = composite_loss(yt3, yp3)
        mae  = float(np.mean(np.abs(yt3-yp3)))
        rmse = float(np.sqrt(np.mean((yt3-yp3)**2)))
        per_img_err = np.mean(np.abs(yt3-yp3), axis=1)     # (10,)
        within_std  = float(np.std(per_img_err, ddof=0))
        per_loop.append(dict(loop=li+1, r2=r2, loss=loss, mae=mae, rmse=rmse, within=within_std))

    r2s   = np.array([p["r2"]   for p in per_loop])
    losses= np.array([p["loss"] for p in per_loop])
    maes  = np.array([p["mae"]  for p in per_loop])
    rmses = np.array([p["rmse"] for p in per_loop])
    sigma = float(np.std(r2s, ddof=1))                     # across-loops σ (sample std)
    se    = sigma/math.sqrt(N_LOOPS)
    margin= Z_95*se
    ci    = (xbar-margin, xbar+margin)

    # Grad-CAM / attention
    if cfg["needs_vit_keras"]:
        attention_for_vit(model, cfg)
    else:
        gradcam_for_cnn(model, cfg)

    # Prediction + Grad-CAM/attention on the COMPETITION test image(s)
    comp_records = comp_test_infer_cam(model, cfg)

    # SHAP + LIME (SLOW) for the 5 fixed images and the competition test image
    if RUN_SHAP_LIME:
        fixed_paths = [test_df.iloc[p]["image_path"] for p in GRADCAM_IDS]
        shap_lime_for_paths(model, cfg, fixed_paths, "fixed_testset")
        shap_lime_for_paths(model, cfg, COMP_TEST_PATHS, "competition_test")

    # ---- build the sheet layout (list of rows) ----
    rows=[]
    rows.append([f"MODEL: {cfg['name']}  (best stage: {cfg['best_stage']})"])
    rows.append([])
    rows.append(["[A] PER-LOOP METRICS"])
    rows.append(["Loop","Images","R2(comp)","Loss","MAE","RMSE","WithinLoopStd"])
    for p in per_loop:
        rows.append([p["loop"], N_PER_LOOP, round(p["r2"],4), round(p["loss"],4),
                     round(p["mae"],4), round(p["rmse"],4), round(p["within"],4)])
    rows.append([])
    rows.append(["[B] SUMMARY ACROSS 15 LOOPS"])
    rows.append(["Metric","Mean","Std"])
    rows.append(["R2",   round(float(r2s.mean()),4),   round(sigma,4)])
    rows.append(["Loss", round(float(losses.mean()),4),round(float(losses.std(ddof=1)),4)])
    rows.append(["MAE",  round(float(maes.mean()),4),  round(float(maes.std(ddof=1)),4)])
    rows.append(["RMSE", round(float(rmses.mean()),4), round(float(rmses.std(ddof=1)),4)])
    rows.append([])
    rows.append(["[C] 95% CONFIDENCE INTERVAL (on R2)"])
    rows.append(["x-bar (full-test R2, G+C)", round(xbar,4)])
    rows.append(["recorded train R2 (ref)",   cfg["recorded_r2"]])
    rows.append(["Z (95%)", Z_95])
    rows.append(["sigma (std of 15 loop R2)", round(sigma,4)])
    rows.append(["n (loops)", N_LOOPS])
    rows.append(["Std Error = sigma/sqrt(n)", round(se,5)])
    rows.append(["Margin = Z*SE", round(margin,5)])
    rows.append(["CI lower", round(ci[0],4)])
    rows.append(["CI upper", round(ci[1],4)])
    rows.append(["RESULT", f"{xbar:.4f} +/- {margin:.4f}"])
    rows.append([])
    rows.append(["[D] IMAGE OVERLAP (shared across models; see loop_images.txt)"])
    rows.append(["Unique images used", f"{OVERLAP['unique']}/{N_TEST}"])
    rows.append(["Images in >1 loop", OVERLAP["in_more_than_one"]])
    rows.append(["Max picks of one image", OVERLAP["max_picks"]])

    result = dict(name=cfg["name"], best_stage=cfg["best_stage"], xbar=xbar, sigma=sigma,
                  se=se, margin=margin, ci=ci, mean_r2=float(r2s.mean()), rows=rows, comp_records=comp_records)

    del model; gc.collect(); tf.keras.backend.clear_session()
    return result

## 9. Run all models

In [ ]:
results = []
for key in MODEL_ORDER:
    cfg = REGISTRY[key]
    print("="*60); print("Processing:", cfg["name"])
    try:
        results.append(evaluate_model(cfg))
    except Exception as e:
        print(f"  !! {cfg['name']} FAILED: {repr(e)}")
        results.append(None)
print("Done. Successful models:", sum(r is not None for r in results))

## 10. Write Excel + txt

In [ ]:
# ---- txt: per-loop image paths + overlap ----
with open(TXT,"w") as f:
    f.write("Image paths sampled per loop (SAME across all models; seed = loop index)\n")
    for i, loop in enumerate(LOOP_PATHS):
        f.write(f"\n--- Loop {i+1} ({len(loop)} images) ---\n")
        for p in loop: f.write(f"  {p}\n")
    f.write("\n=== OVERLAP SUMMARY ===\n")
    f.write(f"Total picks: {OVERLAP['total_picks']} | Unique images: {OVERLAP['unique']}/{N_TEST}\n")
    f.write(f"Images appearing in >1 loop: {OVERLAP['in_more_than_one']}\n")
    f.write(f"Max times one image picked: {OVERLAP['max_picks']}\n")
    f.write("\nImages picked more than once (path: count):\n")
    for p,v in sorted(cnt.items(), key=lambda x:-x[1]):
        if v>1: f.write(f"  {p}: {v}\n")
print("wrote", TXT)

# ---- excel: Summary sheet + one sheet per model ----
def rows_to_df(rows):
    w = max((len(r) for r in rows), default=1)
    return pd.DataFrame([list(r)+[""]*(w-len(r)) for r in rows])

ok = [r for r in results if r is not None]
summary_df = pd.DataFrame([{
    "Model": r["name"], "Best stage": r["best_stage"],
    "x-bar R2 (G+C)": round(r["xbar"],4), "Mean loop R2": round(r["mean_r2"],4),
    "sigma": round(r["sigma"],4), "n": N_LOOPS,
    "Margin (95%)": round(r["margin"],4),
    "CI lower": round(r["ci"][0],4), "CI upper": round(r["ci"][1],4),
    "Result": f"{r['xbar']:.4f} +/- {r['margin']:.4f}",
} for r in ok]).sort_values("x-bar R2 (G+C)", ascending=False)

# competition test-image predictions across models
comp_rows = []
for r in ok: comp_rows.extend(r.get("comp_records", []))
comp_df = pd.DataFrame(comp_rows)
if len(comp_df): comp_df.to_csv("competition_test_predictions.csv", index=False)

with pd.ExcelWriter(XLSX, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    if len(comp_df): comp_df.to_excel(writer, sheet_name="CompTestPred", index=False)
    for r in ok:
        rows_to_df(r["rows"]).to_excel(writer, sheet_name=r["name"][:31], index=False, header=False)
print("wrote", XLSX)
print("\nGrad-CAM/attention PNGs under:", GRADCAM_DIR)
print("wrote competition_test_predictions.csv")
display(summary_df)
if len(comp_df): display(comp_df)